In [1]:
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DATA_DIR = Path("../data/raw")
jobs = pd.read_csv(DATA_DIR / "naukri_com-job_sample.csv")

print(jobs.columns.tolist())
display(jobs.head())

['company', 'education', 'experience', 'industry', 'jobdescription', 'jobid', 'joblocation_address', 'jobtitle', 'numberofpositions', 'payrate', 'postdate', 'site_name', 'skills', 'uniq_id']


,company,education,experience,industry,jobdescription,jobid,joblocation_address,jobtitle,numberofpositions,payrate,postdate,site_name,skills,uniq_id
0,MM Media Pvt Ltd,UG: B.Tech/B.E. - Any Specialization PG:Any Po...,0 - 1 yrs,Media / Entertainment / Internet,Job Description Send me Jobs like this Quali...,210516002263,Chennai,Walkin Data Entry Operator (night Shift),NaN,"1,50,000 - 2,25,000 P.A",2016-05-21 19:30:00 +0000,NaN,ITES,43b19632647068535437c774b6ca6cf8
1,find live infotech,UG: B.Tech/B.E. - Any Specialization PG:MBA/PG...,0 - 0 yrs,Advertising / PR / MR / Event Management,Job Description Send me Jobs like this Quali...,210516002391,Chennai,Work Based Onhome Based Part Time.,60.0,"1,50,000 - 2,50,000 P.A. 20000",2016-05-21 19:30:00 +0000,NaN,Marketing,d4c72325e57f89f364812b5ed5a795f0
2,Softtech Career Infosystem Pvt. Ltd,UG: Any Graduate - Any Specialization PG:Any P...,4 - 8 yrs,IT-Software / Software Services,Job Description Send me Jobs like this - as ...,101016900534,Bengaluru,Pl/sql Developer - SQL,NaN,Not Disclosed by Recruiter,2016-10-13 16:20:55 +0000,NaN,IT Software - Application Programming,c47df6f4cfdf5b46f1fd713ba61b9eba
3,Onboard HRServices LLP,UG: Any Graduate - Any Specialization PG:CA Do...,11 - 15 yrs,Banking / Financial Services / Broking,Job Description Send me Jobs like this - Inv...,81016900536,"Mumbai, Bengaluru, Kolkata, Chennai, Coimbator...",Manager/ad/partner - Indirect Tax - CA,NaN,Not Disclosed by Recruiter,2016-10-13 16:20:55 +0000,NaN,Accounts,115d28f140f694dd1cc61c53d03c66ae
4,Spire Technologies and Solutions Pvt. Ltd.,UG: B.Tech/B.E. - Any Specialization PG:Any Po...,6 - 8 yrs,IT-Software / Software Services,Job Description Send me Jobs like this Pleas...,120916002122,Bengaluru,JAVA Technical Lead (6-8 yrs) -,4.0,Not Disclosed by Recruiter,2016-10-13 16:20:55 +0000,NaN,IT Software - Application Programming,a12553fc03bc7bcced8b1bb8963f97b4


In [2]:
def find_column(columns, candidates):
    lower_columns = {str(col).lower().strip(): col for col in columns}

    for candidate in candidates:
        if candidate.lower() in lower_columns:
            return lower_columns[candidate.lower()]

    for col in columns:
        if any(candidate.lower() in str(col).lower() for candidate in candidates):
            return col

    return None

title_col = find_column(
    jobs.columns,
    ["jobtitle", "job title", "title", "position"]
)

description_col = find_column(
    jobs.columns,
    ["jobdescription", "job description", "description", "job_details"]
)

print("Title column:", title_col)
print("Description column:", description_col)

if description_col is None:
    raise ValueError("No job description column found. Check the printed column names.")

Title column: jobtitle
Description column: jobdescription


In [3]:
jobs = jobs.dropna(subset=[description_col]).copy()
jobs[description_col] = jobs[description_col].astype(str)

if title_col:
    jobs["Display Title"] = jobs[title_col].fillna("Untitled Job").astype(str)
else:
    jobs["Display Title"] = "Untitled Job"

jobs = jobs.reset_index(drop=True)

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=20000
)

job_matrix = vectorizer.fit_transform(jobs[description_col])

def recommend_jobs(resume_text, top_n=5):
    resume_vector = vectorizer.transform([resume_text])
    scores = cosine_similarity(resume_vector, job_matrix).flatten()

    result = jobs.copy()
    result["Match Score (%)"] = (scores * 100).round(2)

    columns = ["Display Title", "Match Score (%)", description_col]
    return result.sort_values(
        "Match Score (%)", ascending=False
    )[columns].head(top_n)

In [4]:
sample_resume = """
Python developer with SQL, pandas, machine learning,
data analysis and problem-solving experience.
"""

recommendations = recommend_jobs(sample_resume, top_n=5)
display(recommendations)

,Display Title,Match Score (%),jobdescription
10860,Sr Scientist/ NLP/ Machine Learning/ Data Mini...,47.45,Job Description Send me Jobs like this Major...
15557,Machine Learning Scientist,45.06,Job Description Send me Jobs like this Major...
17438,Software Engineer - Machine Learning,43.61,Job Description Send me Jobs like this espon...
1348,Software Engineer,42.35,Job Description Send me Jobs like this Come ...
15614,Data Scientist - Machine Learning/nlp,38.90,Job Description Send me Jobs like this roles...


In [5]:
resume_text = input("Paste resume text: ")

if resume_text.strip():
    display(recommend_jobs(resume_text, top_n=5))
else:
    print("Please enter resume text.")

Please enter resume text.
